# 🎨 Notebook 6 – Visualization

**Depends on:** `01_setup.ipynb` + `02_preprocess.ipynb` + `04_pose_train.ipynb`

**What this notebook does:**

| Step | Description |
|------|-------------|
| 6.1 | Load config, model, and helpers |
| 6.2 | Define projection + drawing functions |
| 6.3 | Visualize pose: GT (red) vs Predicted (green) wireframe |
| 6.4 | Visualize YOLO detections on sample images |
| 6.5 | Visualize radius maps (heatmaps) |
| 6.6 | Save all visualizations to Drive |

**How the 3D overlay works:**
1. Load predicted 7-D pose vector `[tx, ty, tz, qx, qy, qz, qw]`
2. Convert to 4×4 matrix: `T = [R | t]`
3. Transform all mesh vertices: `v_cam = R @ v_obj + t`
4. Project to 2D: `px = K @ v_cam / v_cam[2]`
5. Draw triangle edges of the mesh wireframe on the RGB image

## Cell 6.1 – Load config, imports, and model

In [ ]:
import os, json, sys
import numpy as np
import cv2
import torch
import open3d as o3d
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image as PILImage
from torch.cuda.amp import autocast
import torchvision.transforms as T
from scipy.spatial.transform import Rotation as SciR

with open('/content/config.json') as f:
    CONFIG = json.load(f)

DATA_DIR    = CONFIG['DATA_DIR']
ALL_CLASSES = CONFIG['ALL_CLASSES']
CLASS_NAMES = CONFIG['CLASS_NAMES']
CAMERA_K    = np.array(CONFIG['CAMERA_K'], dtype=np.float32)
REPO_DIR    = CONFIG['REPO_DIR']
MODEL_PATH  = CONFIG.get('BEST_POSE_MODEL')
DRIVE_MODELS= CONFIG['DRIVE_MODELS']

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.model   import EnhancedRCVPose
from src.dataset import PoseDataset, read_depth_dpt, pose_matrix_to_vector

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── H100 / Ampere optimisations ───────────────────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.set_float32_matmul_precision('high')
    torch.backends.cudnn.benchmark = True
    IS_H100   = 'H100' in gpu_name or 'H800' in gpu_name
    AMP_DTYPE = torch.bfloat16 if IS_H100 else torch.float16
    print(f'   GPU       : {gpu_name}')
    print(f'   AMP dtype : {AMP_DTYPE}')
else:
    IS_H100   = False
    AMP_DTYPE = torch.float16

pose_model = EnhancedRCVPose().to(DEVICE)
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
state = {k.replace('_orig_mod.', ''): v for k, v in ckpt['model_state_dict'].items()}
pose_model.load_state_dict(state, strict=False)
pose_model.eval()
print(f'✅ Pose model loaded from {MODEL_PATH}')

RGB_TRANSFORM = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('✅ Ready.')

## Cell 6.2 – Helper functions for 3D → 2D projection and drawing

**`pose_vec_to_matrix`:** Converts `[tx, ty, tz, qx, qy, qz, qw]` to a `(4,4)` homogeneous matrix. Translation in metres, quaternion in scipy convention (scalar last).

**`project_vertices`:**
1. Homogenise mesh vertices: `v_h = [x, y, z, 1]`
2. Transform to camera space: `v_c = T @ v_h`
3. Project: `[px, py] = K @ v_c[:3] / v_c[2]`

Returns pixel coords `(N,2)` and visibility mask `(z > 0)`.

**`draw_wireframe`:** Iterates mesh triangles, draws the 3 edges of each visible triangle. Clips triangles where any vertex projects outside image bounds.

**`load_sample`:** Loads one RGB+depth sample and returns GPU-ready tensors + raw numpy arrays needed for visualisation.

In [ ]:
def pose_vec_to_matrix(vec: np.ndarray) -> np.ndarray:
    """7-D [tx,ty,tz, qx,qy,qz,qw] → (4,4) homogeneous transform."""
    M = np.eye(4, dtype=np.float32)
    M[:3, :3] = SciR.from_quat(vec[3:]).as_matrix()
    M[:3,  3] = vec[:3]
    return M


def project_vertices(verts: np.ndarray, T_mat: np.ndarray, K: np.ndarray):
    """
    Project 3D mesh vertices to 2D image plane.
    verts : (N, 3) float32  object-space vertices in metres
    T_mat : (4, 4) float32  model-to-camera transform
    K     : (3, 3) float32  camera intrinsic matrix
    Returns: pts2d (N,2) int, valid (N,) bool
    """
    N    = len(verts)
    h    = np.hstack([verts, np.ones((N, 1), dtype=np.float32)])
    cam  = (T_mat @ h.T).T
    valid = cam[:, 2] > 0
    proj  = (K @ cam[:, :3].T).T
    eps   = 1e-6
    pts2d = (proj[:, :2] / (proj[:, 2:3] + eps)).astype(int)
    return pts2d, valid


def draw_wireframe(img: np.ndarray, pts2d: np.ndarray,
                   faces: np.ndarray, valid: np.ndarray,
                   color=(0, 255, 0), thickness=1) -> np.ndarray:
    """Draw visible mesh wireframe edges onto a copy of img."""
    H, W  = img.shape[:2]
    out   = img.copy()
    for tri in faces:
        if not valid[tri[0]] or not valid[tri[1]] or not valid[tri[2]]:
            continue
        pts = pts2d[tri]
        if not all(0 <= p[0] < W and 0 <= p[1] < H for p in pts):
            continue
        cv2.line(out, tuple(pts[0]), tuple(pts[1]), color, thickness)
        cv2.line(out, tuple(pts[1]), tuple(pts[2]), color, thickness)
        cv2.line(out, tuple(pts[2]), tuple(pts[0]), color, thickness)
    return out


def load_sample(cls: str, sample_id: str):
    """
    Load one sample. Returns:
        rgb_raw  : (H,W,3) uint8 numpy
        rgb_t    : (1,3,H,W) GPU tensor (normalised)
        depth_t  : (1,1,H,W) GPU tensor (metres)
        gt_vec   : (7,) float32 ground-truth pose
    """
    cls_dir = os.path.join(DATA_DIR, cls)
    rgb_bgr = cv2.imread(os.path.join(cls_dir, 'rgb', f'{sample_id}.png'))
    rgb_raw = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2RGB)
    rgb_t   = RGB_TRANSFORM(PILImage.fromarray(rgb_raw)).unsqueeze(0).to(DEVICE)
    depth_np = read_depth_dpt(os.path.join(cls_dir, 'depth', f'{sample_id}.dpt')) / 1000.0
    depth_t  = torch.from_numpy(depth_np).unsqueeze(0).unsqueeze(0).to(DEVICE)
    gt_vec = pose_matrix_to_vector(
        np.load(os.path.join(cls_dir, 'pose', f'pose{sample_id}.npy'))
    )
    return rgb_raw, rgb_t, depth_t, gt_vec


print('✅ Helper functions defined.')

## Cell 6.3 – Visualize pose: GT (red) vs Predicted (green) wireframe

For each class in `VIS_CLASSES`:
1. Load the first sample from the test split
2. Run model inference → predicted 7-D pose
3. Load GT pose for comparison
4. Draw GT wireframe in **red** and predicted wireframe in **green**
5. Show side-by-side: Original | GT Overlay | Predicted Overlay
6. Print numerical errors below the figure

Change `VIS_CLASSES` to select which objects to visualise. Set `VIS_CLASSES = ALL_CLASSES` to visualise all 13 objects.

In [ ]:
VIS_CLASSES = ['01', '05', '08', '09']    # ← change as desired

for cls in VIS_CLASSES:
    cls_dir  = os.path.join(DATA_DIR, cls)
    split_f  = os.path.join(cls_dir, 'Split', 'test.txt')
    mesh_ply = os.path.join(cls_dir, 'mesh.ply')

    if not os.path.isfile(split_f):
        print(f'⚠️  Class {cls}: Split/test.txt not found')
        continue
    if not os.path.isfile(mesh_ply):
        print(f'⚠️  Class {cls}: mesh.ply not found')
        continue

    with open(split_f) as f:
        sample_id = f.readline().strip()

    mesh  = o3d.io.read_triangle_mesh(mesh_ply)
    verts = np.asarray(mesh.vertices, dtype=np.float32)
    faces = np.asarray(mesh.triangles)
    if verts.max() > 1.0:
        verts /= 1000.0    # mm → m

    rgb_raw, rgb_t, depth_t, gt_vec = load_sample(cls, sample_id)

    with torch.no_grad():
        with autocast(dtype=AMP_DTYPE):
            pred_pose, _ = pose_model(rgb_t, depth_t)
    pred_vec = pred_pose.float().cpu().numpy()[0]

    T_pred = pose_vec_to_matrix(pred_vec)
    T_gt   = pose_vec_to_matrix(gt_vec)
    pts_p, vp = project_vertices(verts, T_pred, CAMERA_K)
    pts_g, vg = project_vertices(verts, T_gt,   CAMERA_K)
    img_pred = draw_wireframe(rgb_raw, pts_p, faces, vp, color=(0, 255, 0), thickness=1)
    img_gt   = draw_wireframe(rgb_raw, pts_g, faces, vg, color=(255, 0, 0), thickness=1)

    t_err = float(np.linalg.norm(pred_vec[:3] - gt_vec[:3])) * 100   # cm
    q_p   = pred_vec[3:] / (np.linalg.norm(pred_vec[3:]) + 1e-9)
    q_g   = gt_vec[3:]   / (np.linalg.norm(gt_vec[3:])   + 1e-9)
    r_err = float(np.degrees(2.0 * np.arccos(np.clip(abs(np.dot(q_p, q_g)), 0, 1-1e-7))))

    cls_name = CLASS_NAMES[ALL_CLASSES.index(cls)]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'Class {cls} – {cls_name}   |   Sample {sample_id}',
                 fontsize=13, fontweight='bold')

    for ax, img, title in zip(axes,
                              [rgb_raw, img_gt, img_pred],
                              ['Original RGB',
                               'GT Pose  (red wireframe)',
                               'Predicted Pose  (green wireframe)']):
        ax.imshow(img); ax.set_title(title, fontsize=11); ax.axis('off')

    info = (f'Translation error: {t_err:.1f} cm   |   Rotation error: {r_err:.1f}°   |   '
            f'Pred t: [{pred_vec[0]*100:.1f}, {pred_vec[1]*100:.1f}, {pred_vec[2]*100:.1f}] cm   '
            f'GT t: [{gt_vec[0]*100:.1f}, {gt_vec[1]*100:.1f}, {gt_vec[2]*100:.1f}] cm')
    fig.text(0.5, -0.01, info, ha='center', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    plt.tight_layout()
    plt.show()
    print()

## Cell 6.4 – Visualize YOLO detections on sample images

We load the best YOLO model (saved by `03_yolo_train.ipynb`) and run inference on 6 random validation images.

`ultralytics model.predict()` returns a `Results` object. `results[0].plot()` returns a BGR numpy array with boxes drawn. We convert to RGB and display with matplotlib.

In [ ]:
import glob
import random
from ultralytics import YOLO

yolo_path = CONFIG.get('YOLO_MODEL_PATH', os.path.join(DRIVE_MODELS, 'yolo_best.pt'))
yolo_model = YOLO(yolo_path)
print(f'✅ YOLO model loaded: {yolo_path}')

YOLO_DIR    = CONFIG['YOLO_DIR']
val_images  = glob.glob(os.path.join(YOLO_DIR, 'images/val/*.png'))
random.seed(7)
samples = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('YOLOv8 Detections on Validation Images', fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flat, samples):
    results = yolo_model.predict(img_path, conf=0.25, verbose=False)
    drawn   = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)
    fname   = os.path.basename(img_path)
    ax.imshow(drawn); ax.set_title(fname, fontsize=8); ax.axis('off')

plt.tight_layout()
plt.savefig('/content/yolo_detections.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ YOLO detection grid saved to /content/yolo_detections.png')

## Cell 6.5 – Visualize radius maps as heatmaps

Each radius map stores per-pixel 3D distance (metres) to one keypoint. We show the 9 radius maps for a single sample side-by-side.

**Colour scale:**
- 🟦 Dark blue = small distance (pixel is close to the keypoint in 3D)
- 🟡 Yellow = large distance
- ⬛ Black/grey background = masked (depth = 0, not part of the object)

**Two rows:**
- Row 1 (green border): **Ground-truth** radius maps from data preprocessing
- Row 2 (blue border): **Predicted** radius maps from the model

Change `VIS_CLS_RM` to select a different class.

In [ ]:
VIS_CLS_RM = '01'

cls_dir   = os.path.join(DATA_DIR, VIS_CLS_RM)
split_f   = os.path.join(cls_dir, 'Split', 'test.txt')
with open(split_f) as f:
    sid = f.readline().strip()

gt_rmaps = []
for pt in range(1, 10):
    rmap_path = os.path.join(cls_dir, f'Out_pt{pt}_dm', f'{sid}.npy')
    gt_rmaps.append(np.load(rmap_path))

rgb_raw, rgb_t, depth_t, gt_vec = load_sample(VIS_CLS_RM, sid)
with torch.no_grad():
    with autocast(dtype=AMP_DTYPE):
        _, pred_rmaps = pose_model(rgb_t, depth_t)
pred_rmaps = pred_rmaps.float().cpu().numpy()[0]    # (9, H, W)

fig, axes = plt.subplots(2, 9, figsize=(22, 5))
fig.suptitle(f'Radius Maps — Class {VIS_CLS_RM}  Sample {sid}', fontsize=12)

for col in range(9):
    ax = axes[0, col]
    ax.imshow(gt_rmaps[col], cmap='plasma')
    ax.set_title(f'GT kp{col+1}', fontsize=8)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('green'); spine.set_linewidth(2)

    ax = axes[1, col]
    ax.imshow(pred_rmaps[col], cmap='plasma')
    ax.set_title(f'Pred kp{col+1}', fontsize=8)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('dodgerblue'); spine.set_linewidth(2)

axes[0, 0].set_ylabel('GT', fontsize=10, color='green')
axes[1, 0].set_ylabel('Pred', fontsize=10, color='dodgerblue')

plt.tight_layout()
plt.savefig('/content/radius_maps.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Radius maps saved to /content/radius_maps.png')

## Cell 6.6 – Save all visualizations to Google Drive

We copy the three generated files to Drive so they are accessible after the Colab session ends:
- `evaluation_results.png` — bar charts from `05_evaluate.ipynb`
- `yolo_detections.png` — YOLO detection grid
- `radius_maps.png` — radius map GT vs. predicted

In [ ]:
import shutil

viz_dir = os.path.join(DRIVE_MODELS, 'visualizations')
os.makedirs(viz_dir, exist_ok=True)

files = [
    '/content/evaluation_results.png',
    '/content/yolo_detections.png',
    '/content/radius_maps.png',
]

for src in files:
    if os.path.isfile(src):
        dst = os.path.join(viz_dir, os.path.basename(src))
        shutil.copy(src, dst)
        print(f'   ✅ {os.path.basename(src)} → {dst}')
    else:
        print(f'   ⚠️  {src} not found (run the corresponding cell first)')

print(f'\n✅ All visualizations saved to {viz_dir}')

---
## ✅ Pipeline Complete!

You have successfully run the full 6D pose estimation pipeline:

| Notebook | Status |
|----------|--------|
| `01_setup.ipynb` | ✅ Environment set up, dataset extracted |
| `02_preprocess.ipynb` | ✅ Poses, keypoints, radius maps, splits prepared |
| `03_yolo_train.ipynb` | ✅ YOLOv8s trained for object detection |
| `04_pose_train.ipynb` | ✅ EnhancedRCVPose trained (stage 1 + 2 + fine-tune) |
| `05_evaluate.ipynb` | ✅ Validation and test metrics computed |
| `06_visualize.ipynb` | ✅ Pose overlay, YOLO detections, radius maps visualised |

All outputs are saved to **Google Drive** under `models/`.